# Kaggriculture Meta Census — Lineage Occupancy

Who is farming what on the top of the ladder, and how fast does the meta rotate?

This notebook tracks **agent lineages** in the official
[Kaggriculture episodes datasets](https://www.kaggle.com/datasets/kaggle/kaggriculture-episodes-index)
(a daily excerpt of top-band games) and charts their **occupancy over time**.

## Method

- A seat's **lineage** is the SHA-256 (first 10 hex chars) of its first 24 steps
  of actions — the "opening signature". Deterministic agents produce a stable
  signature across towns and seeds, so identical-opening agents cluster
  together even when their mid-game behavior adapts.
- **Occupancy** = share of seats per day carrying each signature.
- A lineage is **labeled with a public notebook's name only when that
  notebook's extracted agent reproduces the signature in a local mirror match**
  (a verifiable claim anyone can replicate). Everything else stays a bare hash
  — no guessing about private agents' origins.
- Earlier days are carried as pre-computed counts (same pipeline, run daily);
  the attached day is mined live from the raw episode JSONs below.

The result is a picture of the meta as an ecosystem: public-notebook waves
sweeping the ladder in days, counter-strategies rising, and the occasional
unlabeled newcomer climbing out of nowhere.


In [ ]:
# Kaggriculture meta census (public edition).
#
# Mines the attached daily episodes dataset into a per-seat lineage census,
# merges it with the carried history, and renders the occupancy chart inline.
# Outputs:
#   census-<date>.json  — one row per episode: town, rewards, teams, lineages
#   census_trend.html   — the same interactive chart as a standalone file

import glob
import hashlib
import json
import os
import re

# Lineages labeled by their source notebook — ONLY where the notebook's
# extracted agent reproduces the opening signature in a local mirror match.
# Everything else is shown as its bare signature hash.
NOTEBOOK_LABELS = {
    "bb7fbea029": "v54 Strict Flow (Kaito Fukami)",
    "ee954925d4": "Shape the Shop Work the Pasture (Indar Karhana)",
    "cb8e6be117": "Multi-Route Farming Agent (flexonafft)",
}

# Daily seat counts per lineage from previous runs of this same pipeline.
# --- nbmine:history:start ---
HISTORY = {"2026-08-23": {"0336bbe1df": 134, "040ffdf2ed": 27, "2b01a546d4": 90, "2f54686340": 40, "393c72dd52": 35, "47a038ebea": 4, "5146031dc6": 24, "62c67bbd0d": 20, "758d7736d0": 60, "7e8222dcd3": 4, "8786c0d4da": 38, "95f4f78d90": 59, "af2d00c1a8": 7, "baa3f76dbf": 311, "c831787799": 20, "cb8e6be117": 478, "e66b86ee81": 12, "e838598664": 9, "ffd5f2f569": 12}, "2026-08-24": {"0336bbe1df": 48, "09970d9f67": 25, "2b01a546d4": 50, "2f54686340": 36, "5146031dc6": 20, "62c67bbd0d": 1, "758d7736d0": 7, "8786c0d4da": 5, "9410a69601": 1, "95f4f78d90": 16, "baa3f76dbf": 1024, "c831787799": 24, "cb8e6be117": 121, "e66b86ee81": 3, "ffd5f2f569": 3}, "2026-08-25": {"0336bbe1df": 6, "09970d9f67": 34, "2b01a546d4": 48, "2f54686340": 29, "50bead8d1b": 10, "5146031dc6": 37, "62c67bbd0d": 1, "95f4f78d90": 4, "99f2f6f4c6": 10, "9c096cd3cf": 3, "baa3f76dbf": 1102, "c2ebeb3bf2": 18, "c831787799": 22, "cb8e6be117": 7, "f30d2561bb": 43, "ffd5f2f569": 2}, "2026-08-26": {"09970d9f67": 24, "0a33371cdf": 1, "2b01a546d4": 45, "2b3268ddc5": 10, "2f54686340": 29, "50bead8d1b": 94, "5146031dc6": 55, "9410a69601": 1, "97d88078fd": 3, "99f2f6f4c6": 32, "9c096cd3cf": 11, "baa3f76dbf": 827, "bb7fbea029": 12, "be0a8cd059": 81, "c2ebeb3bf2": 52, "c831787799": 17, "c9713d0283": 2, "f30d2561bb": 68, "ffd5f2f569": 10}, "2026-08-27": {"09970d9f67": 33, "0a33371cdf": 8, "10ac811f26": 1, "12a91d93c6": 2, "2b01a546d4": 36, "2b3268ddc5": 7, "2f54686340": 35, "50bead8d1b": 492, "5146031dc6": 38, "77f1277f9a": 2, "889de90d84": 44, "8a94a2d4a7": 55, "97d88078fd": 3, "99f2f6f4c6": 34, "9c096cd3cf": 48, "baa3f76dbf": 333, "bb7fbea029": 69, "be0a8cd059": 30, "c2ebeb3bf2": 29, "c831787799": 9, "d3ef5fb203": 15, "f30d2561bb": 55, "ffd5f2f569": 14}, "2026-08-28": {"09970d9f67": 34, "0a33371cdf": 2, "1099c6ffc7": 12, "10ac811f26": 1, "12a91d93c6": 5, "2b01a546d4": 1, "2d9e4b260e": 5, "2f54686340": 36, "42ed29929f": 32, "50bead8d1b": 215, "5146031dc6": 41, "77f1277f9a": 15, "889de90d84": 14, "8a94a2d4a7": 78, "97d88078fd": 23, "99f2f6f4c6": 42, "9e6e9a13f0": 55, "aca7211231": 10, "af2d00c1a8": 1, "b12654cf75": 26, "baa3f76dbf": 94, "bb7fbea029": 477, "be0a8cd059": 3, "c2ebeb3bf2": 8, "c831787799": 26, "c9713d0283": 9, "d2b815f0c3": 57, "d3ef5fb203": 4, "f30d2561bb": 18, "ffd5f2f569": 8}, "2026-08-29": {"09970d9f67": 39, "2d9e4b260e": 1, "2f54686340": 1, "42ed29929f": 7, "50bead8d1b": 26, "5146031dc6": 35, "77f1277f9a": 21, "891495828b": 54, "8a94a2d4a7": 9, "99f2f6f4c6": 22, "9e6e9a13f0": 7, "b12654cf75": 48, "baa3f76dbf": 6, "bb7fbea029": 774, "be0a8cd059": 1, "c831787799": 31, "c9713d0283": 9, "d2b815f0c3": 60, "ee954925d4": 76, "f30d2561bb": 104, "ffd5f2f569": 3}, "2026-08-30": {"02f2165577": 10, "09970d9f67": 12, "42ed29929f": 15, "5146031dc6": 36, "548839bf65": 1, "5a5bb07cde": 29, "77f1277f9a": 9, "891495828b": 212, "a8290df987": 2, "b12654cf75": 33, "b93635aa37": 12, "bb7fbea029": 509, "c831787799": 25, "c9713d0283": 10, "d2b815f0c3": 43, "e7f89965a4": 3, "e9c9700032": 96, "ee954925d4": 192, "f30d2561bb": 79}, "2026-08-31": {"02f2165577": 19, "1f6a8a5ba5": 1, "42ed29929f": 21, "5146031dc6": 9, "548839bf65": 7, "5a5bb07cde": 62, "7529e16a57": 41, "891495828b": 220, "9e6e9a13f0": 30, "a8290df987": 55, "b12654cf75": 191, "b93635aa37": 13, "bb7fbea029": 95, "c831787799": 22, "e707c6c560": 2, "e9c9700032": 384, "ee954925d4": 68, "f30d2561bb": 90}, "2026-09-01": {"15d5638b8b": 41, "198d4ef8c5": 1, "1fa5c13552": 2, "407677684b": 3, "4a9a71b75c": 3, "5a5bb07cde": 83, "5ccb3e7419": 1, "7529e16a57": 44, "7a32851180": 114, "891495828b": 139, "8cf2dce726": 1, "96d6a4d4f1": 4, "9e6e9a13f0": 94, "a8290df987": 44, "b12654cf75": 245, "b64a38a635": 30, "b7d0048499": 24, "bb7fbea029": 202, "c831787799": 18, "cd7c22f878": 3, "e023a4729d": 21, "e9c9700032": 193, "f30d2561bb": 11, "fbb2028994": 7}, "2026-09-02": {"0f84ed7ca9": 7, "15d5638b8b": 126, "1ed64ee8b6": 3, "407677684b": 3, "4a66ccad44": 102, "4a9a71b75c": 3, "5167ef9b2e": 5, "572f448235": 2, "5a5bb07cde": 15, "7529e16a57": 18, "7584ae92d2": 37, "7a32851180": 15, "82a3c72c31": 411, "8cf2dce726": 106, "921500c57a": 62, "9e6e9a13f0": 52, "a12bc717bd": 5, "a2113ce935": 44, "a3d541669f": 19, "ad83b72700": 14, "af2d001140": 15, "b7d0048499": 156, "bde878b74a": 34, "be07a50e8c": 9, "c6fdb6c947": 38, "cbfd9c5bd8": 12, "eb04e30493": 5, "f99f4470a7": 1, "fbb2028994": 1}, "2026-09-03": {"02b953cf3f": 5, "0d2053e2f3": 23, "15ebeb5076": 15, "206e0ca52b": 16, "2a9aa2f7b6": 64, "313fdee774": 22, "4853c19a6d": 1, "4a66ccad44": 8, "567b3c1670": 89, "572f448235": 97, "6af7240248": 16, "7584ae92d2": 24, "801d626ba6": 76, "82a3c72c31": 25, "920558c65e": 24, "921500c57a": 110, "9e6e9a13f0": 95, "a2113ce935": 25, "ad83b72700": 2, "b17f601b80": 1, "b1bed42093": 225, "b7d0048499": 89, "b9806b1857": 65, "bb7fbea029": 1, "c6fdb6c947": 111, "cbfd9c5bd8": 14, "d7ab0eb7a8": 8, "e3ce37726f": 1, "e8b43f4151": 17, "e9916a8a7b": 8, "f336a49e19": 43, "f866a404d2": 18}, "2026-09-04": {"02b953cf3f": 17, "0d2053e2f3": 99, "15d5638b8b": 47, "15ebeb5076": 61, "206e0ca52b": 238, "2a9aa2f7b6": 1, "313fdee774": 71, "4853c19a6d": 89, "567b3c1670": 3, "572f448235": 16, "630f595e62": 11, "6737318541": 1, "67dd108dc6": 8, "6af7240248": 40, "6e7e6b506f": 13, "751741f2aa": 3, "7dafab486a": 1, "801d626ba6": 125, "8663bc481a": 20, "8a6fb90873": 16, "921500c57a": 19, "9e4772a04e": 1, "9e6e9a13f0": 146, "a2113ce935": 35, "b1bed42093": 36, "b7d0048499": 74, "b9806b1857": 3, "bb7fbea029": 2, "c0ddc7a55d": 1, "c199b8d99c": 1, "ca72492e70": 8, "cc587fe77e": 3, "d470b908f3": 2, "d7ab0eb7a8": 30, "d95b21c65e": 86, "e9916a8a7b": 9}, "2026-09-05": {"02b953cf3f": 11, "0cc9495557": 42, "0ee407b42a": 1, "1305c9267e": 1, "15d5638b8b": 137, "15ebeb5076": 24, "2029546c36": 9, "206e0ca52b": 236, "2ef23e0c6e": 69, "313fdee774": 8, "4853c19a6d": 65, "509124e958": 1, "56d01571c4": 2, "5a3af96ed2": 7, "630f595e62": 12, "6b77ea5fff": 19, "6e7e6b506f": 38, "74fee8f305": 29, "751741f2aa": 17, "773fc1dc20": 1, "801d626ba6": 56, "864d69b2c9": 6, "891495828b": 91, "89f66cc8f0": 4, "8a6fb90873": 2, "921500c57a": 14, "92d4bd36a9": 1, "9949c651af": 1, "9e4772a04e": 26, "a36d6bd8ff": 45, "a9f3eb7993": 3, "b1bed42093": 4, "b9806b1857": 5, "bb7fbea029": 32, "c199b8d99c": 1, "ca72492e70": 34, "cf45f8d309": 1, "d24b793042": 102, "d7ab0eb7a8": 3, "d95b21c65e": 143, "e4d1dcd188": 24, "ef0a639803": 1}, "2026-09-06": {"02b953cf3f": 13, "0987ccc1cf": 1, "0cc9495557": 173, "0d7a70228a": 29, "0e5dce2d95": 63, "0ee407b42a": 158, "0f76e5a717": 1, "15d5638b8b": 12, "1878c75f8b": 8, "1daf3a51f6": 9, "206e0ca52b": 26, "2519e4b820": 4, "27e2775248": 2, "2ef23e0c6e": 31, "333016335f": 66, "368452c2f0": 15, "390c1f4ce0": 2, "3f7aa60113": 20, "448d220e4a": 1, "4853c19a6d": 47, "53d692273b": 5, "56d01571c4": 43, "5a3af96ed2": 53, "5bdb79ed02": 2, "6ba133b6a7": 1, "6d0ab02ebf": 34, "6e7e6b506f": 58, "773fc1dc20": 2, "8041ec8e4a": 16, "8338da50ac": 23, "864d69b2c9": 2, "86c798fec9": 1, "89f66cc8f0": 5, "8a6fb90873": 5, "8b230b25e9": 60, "9482924619": 1, "96eb9e38d4": 1, "98895ba5f7": 2, "9ae83b19d2": 8, "9e4772a04e": 1, "a36d6bd8ff": 43, "a47498b650": 1, "a9f3eb7993": 1, "b9806b1857": 19, "bafed64435": 1, "cf45f8d309": 1, "d24b793042": 75, "d6aae107c3": 1, "d95b21c65e": 15, "da9d90b9e5": 13, "dec95004aa": 5, "defd41a32d": 1, "e4d1dcd188": 134, "f0911bc98e": 1, "f0995b3733": 2, "f18b646c68": 2, "f1e8f23f2d": 4, "f71b6ee2bf": 1}, "2026-09-07": {"00c0569837": 1, "02b953cf3f": 2, "0a5072d329": 1, "0aa21b1ce1": 1, "0b29172b83": 1, "0c9050e6e8": 87, "0cc9495557": 119, "0d7a70228a": 29, "0e5dce2d95": 7, "0ee407b42a": 5, "101124ac7b": 1, "12b34e431d": 1, "131b8b5e64": 1, "13449d6a6c": 1, "156e51c11d": 1, "1878c75f8b": 3, "19b0207422": 1, "1a2fec0e2a": 1, "1a93eb7243": 2, "1c6f52af86": 26, "1d04ee73a6": 8, "1da80acbc9": 1, "1dd28fc672": 1, "200885e89b": 1, "206e0ca52b": 56, "244f5ff8b2": 1, "2476a78a3c": 1, "27a31d076e": 1, "28b88e14f8": 1, "2cbb45b9a1": 2, "2e565edfc4": 1, "2ef23e0c6e": 83, "2fb8ee8ea5": 1, "32962dc1c7": 1, "333016335f": 123, "368452c2f0": 36, "37a7ebdb6c": 15, "3879aa3417": 1, "3d762a2fea": 1, "3da879dfb6": 1, "3fea148534": 9, "419d6d94b3": 1, "41ed7cf626": 1, "4299237a91": 1, "448d220e4a": 2, "450f701b1e": 11, "47b62409f4": 73, "49883e3be0": 1, "4bedc42ef4": 1, "4e6477534b": 1, "50617db969": 1, "509e1ac27a": 16, "53d692273b": 1, "54003103b0": 1, "547061d261": 1, "558ad6d691": 1, "56f2ea569b": 4, "57eb3b96bf": 1, "59fa20a661": 12, "5c198fb9a1": 9, "5d08ed2dc1": 1, "6149893e83": 1, "63415ecbb3": 1, "66e26b5b24": 1, "69d4681599": 40, "6b083ca11b": 1, "6c274c25b1": 5, "6ccb88b8e5": 1, "6d0ab02ebf": 8, "6df9b947ed": 1, "6e7e6b506f": 19, "7105b61ba0": 1, "7156e1efc4": 1, "72a46dcec6": 1, "738760eb5f": 2, "74a90ab195": 1, "74fee8f305": 22, "75b20ce196": 1, "78de292462": 1, "79ae92a7dd": 1, "7a16592df6": 1, "7a23d8277a": 1, "7b8419ffbf": 1, "8041ec8e4a": 3, "82776e8350": 1, "830d54e223": 1, "8338da50ac": 16, "84533056d0": 1, "8600268119": 1, "8aa4799e37": 1, "8b230b25e9": 6, "8ca0912c34": 1, "8ccdffe3ab": 1, "8fba47d610": 1, "9080b4023d": 2, "9333e4162a": 1, "93902ab4e4": 3, "96eb9e38d4": 3, "9ae83b19d2": 1, "9c471e7ba5": 1, "a0f289a3cc": 5, "a10ca980f0": 1, "a2b99498ff": 2, "a31ae5075c": 1, "a33eee4090": 1, "a5a2820bc7": 1, "a60dd8d89e": 1, "a77001f643": 10, "a77b5282d6": 3, "aa2845142d": 1, "ab769c6d23": 1, "ae0ce0c481": 2, "b225dd8bc4": 1, "b550678f1c": 7, "b9b0bb7ffa": 1, "bc83c36a6c": 1, "bcfabf51f4": 1, "bfa2260dcf": 1, "c1bc697268": 1, "c2a9350836": 1, "c6fee349c7": 1, "c841881ff0": 1, "c8c29f601c": 15, "c93184ab25": 12, "cab00bb01b": 1, "caf0dd2178": 1, "cbf5483003": 2, "ce0b4dbba5": 1, "d0285d51c9": 23, "d0c7179fad": 1, "d11389780a": 4, "d14ecdd449": 1, "d22b0fe87e": 4, "d2a2c4d3f4": 1, "d50b3e1e1e": 188, "d6aae107c3": 50, "d727aa6dcb": 1, "dc3513ec77": 1, "df5c733e77": 1, "e1531db47e": 2, "e2067f086c": 1, "e38ae95247": 3, "e4d1dcd188": 21, "e5cf88f297": 1, "ea0f7e1972": 1, "eabbd749cb": 1, "ee093db998": 1, "f013ec86dd": 1, "f08b5ae2e6": 1, "f308e4391f": 1, "f425d4a4c4": 1, "f81eda0cf3": 1, "f959591c10": 1, "fc10816b55": 3, "fd522e2038": 1, "fe176fd5fe": 1}, "2026-09-08": {"006d5240fc": 1, "00d8963aa4": 1, "01e068be23": 1, "034e967bbe": 1, "038a9cebf2": 1, "046ec3a580": 1, "04bf32e027": 1, "05400c3339": 1, "0853adcbbd": 1, "08cd17cdbe": 1, "08ebcb2686": 1, "09106e474d": 1, "09295ff018": 5, "098b1fb007": 1, "0a586d6ab1": 1, "0ad6d3a862": 1, "0b97a379b0": 1, "0bae9ed07f": 1, "0bbe9bf457": 1, "0c9050e6e8": 8, "0cc9495557": 106, "0e5dce2d95": 27, "0ecfe8fe1a": 1, "0ee407b42a": 92, "10605a8512": 1, "108f9ff76c": 1, "10c003c20e": 28, "11824848d9": 1, "125649bd62": 1, "13cae28562": 1, "156ccd3937": 1, "168b36a0a5": 63, "1a31b81b96": 1, "1ab285b1d4": 1, "1ac3beb6ec": 1, "1bf23488ea": 1, "1c6f52af86": 2, "1d90abfd52": 1, "1f9c81945e": 1, "200c2f8126": 1, "206e0ca52b": 111, "21dc3d11d2": 49, "22b196fd00": 1, "2a2bf28827": 1, "2b027439fe": 1, "2bf2369ca9": 1, "2dd763f15e": 1, "2ef23e0c6e": 59, "30583a1b6a": 1, "31551f9b9f": 1, "316fc912ff": 1, "32c89d5931": 1, "333016335f": 58, "39a019af2b": 1, "39b50612fc": 1, "3c20a1704e": 124, "3c392e1d88": 7, "3d2ac63384": 1, "3e03bf02d9": 1, "3e423d44f0": 1, "3e60fc70a9": 1, "3f1a830417": 1, "3f49a70cc6": 1, "3f712a197d": 1, "3fcbfff65f": 1, "3fea148534": 1, "4025622772": 1, "40c243fa07": 1, "419d6d94b3": 2, "43fba7b085": 1, "4507af0ca7": 1, "454f6f6130": 1, "46811864e6": 4, "4769f477a1": 1, "47b62409f4": 118, "482c72a0a1": 1, "4b2554993b": 1, "4bf932695d": 1, "4dbc623ba2": 1, "4f98211f13": 1, "4fe80d34b4": 1, "509e1ac27a": 2, "51049bca43": 1, "550260efd6": 1, "5577c5eb4c": 1, "59c87bbe4b": 1, "59fa20a661": 5, "5c198fb9a1": 14, "5c7f7a4568": 1, "5ca7385588": 1, "5d04fc58da": 1, "5e78b48dbd": 1, "5eaeff4429": 1, "5f3495aca9": 1, "611edb49ea": 1, "62521f08f8": 1, "63501aba35": 1, "6366e6dc04": 1, "650d980016": 1, "656ce4b364": 1, "67aee07adf": 1, "696c3c994c": 1, "6970378224": 1, "69d4681599": 31, "6b0606e002": 1, "6b738eb77b": 1, "6e6afb98b0": 1, "6e7e6b506f": 1, "70537e2ad5": 1, "724b70d266": 1, "73c50224d2": 1, "740eb5e357": 1, "74244cab40": 1, "763fae8218": 1, "7805cea695": 1, "78be7ef4cc": 1, "7a1b5a87d6": 1, "7aca6bf5c1": 1, "7d0eff11ee": 1, "7e9f946c39": 1, "802ed4a7af": 1, "8338da50ac": 34, "84f6d4007c": 1, "85525910a1": 1, "8683fac226": 1, "8727851367": 1, "874df10ca2": 1, "876dc10c8f": 2, "87a3d704fc": 1, "87c801224c": 1, "87d5b262cb": 1, "8a2a8e6346": 1, "8aa4799e37": 2, "8b230b25e9": 26, "8ba959e44c": 1, "9080b4023d": 2, "918f1a3e1f": 1, "92063d7bcd": 2, "921eb4ad4a": 1, "943147b925": 1, "964eb85939": 1, "9666dccdb9": 1, "970c89b9f0": 1, "9966795d4d": 1, "99e6c7269e": 1, "9aef5eda56": 1, "9bc98147dd": 1, "9bd6c8123b": 1, "9f75bad2ef": 1, "a02b43ad4c": 1, "a031e56752": 1, "a1801f59e1": 1, "a19347679e": 2, "a2b99498ff": 1, "a3dd4a7045": 1, "a49ce45536": 1, "a5a2820bc7": 78, "a771176165": 1, "a9dd9fa7a8": 1, "abf8bda88b": 1, "ad121cc328": 1, "af30284832": 1, "b1eeb1cee4": 1, "b231ef8383": 1, "b48d250719": 1, "b550678f1c": 1, "b7c4fa135f": 1, "b9c5a17f09": 1, "bd0ad1d79b": 1, "bf14441e6e": 1, "c0f4e490b0": 1, "c64fa6ffd3": 1, "c69b2e6c4d": 1, "c6f85497b6": 1, "c893d22275": 1, "c9106102f5": 1, "c93184ab25": 21, "c9a03adc69": 1, "ca3d88b17a": 1, "cec5f240d9": 1, "d042574563": 1, "d144a6a568": 1, "d22b0fe87e": 7, "d251a9f51b": 1, "d2646f8ada": 1, "d389501ab8": 1, "d3d436d350": 1, "d50b3e1e1e": 19, "d5e1d55e96": 1, "d6a90a31a1": 1, "d6aae107c3": 4, "d8835364ff": 1, "d891ff04bc": 1, "daeb8da2d5": 1, "dc5df1ef42": 1, "dec5c2ad91": 1, "df1181858a": 1, "df7fd85733": 1, "e052bf3b01": 1, "e0aa42a5c7": 1, "e1531db47e": 4, "e165cf6fb4": 1, "e21da135b6": 1, "e28725dbb7": 1, "e4469c84eb": 1, "e4a0be0107": 1, "e521cac8df": 1, "e693da474e": 9, "e70f18fef0": 1, "e7c3667e83": 1, "e7e9f248e6": 1, "e98668601e": 1, "eb5b581001": 1, "ebb6fa8ec0": 1, "ec4dae741c": 1, "ec8631a864": 1, "ee643db8dc": 4, "ef679e6518": 1, "efc9262034": 1, "f237969943": 1, "f3100511dc": 1, "f3e422cdb6": 1, "f48e74d280": 1, "f5fc2e31cf": 1, "f7a17704a0": 1, "f8171bd992": 1, "fa007a87c4": 3, "fe21e9bdc3": 1, "ffd0c7f803": 1}, "2026-09-09": {"009b3bca0d": 39, "00b181e768": 1, "0498bbe4b9": 1, "06d24527c6": 2, "07725b5ed3": 1, "0839753672": 46, "084f03fd33": 1, "0cafdaad46": 1, "0cc9495557": 74, "0ee407b42a": 47, "0f2bb14edf": 1, "10c003c20e": 1, "11bcfd0449": 1, "12ae57cb86": 1, "12e8adaaef": 1, "131c518eef": 1, "151bbb00c0": 1, "168b36a0a5": 86, "17e11a403e": 1, "186d67f3be": 1, "1977296ebf": 15, "1ad4580ea6": 1, "1b360c204e": 1, "1b7a9ee0fe": 1, "1d0b8de634": 2, "1e75ed8443": 1, "206e0ca52b": 21, "20df1fcf25": 1, "2186e7ab7f": 9, "21dc3d11d2": 23, "2278642bc4": 1, "2300ab1706": 7, "23b5a57ff7": 34, "2532b46014": 1, "25d05e0895": 1, "281e031004": 2, "28c7b99dc1": 1, "28c9ccaae4": 1, "29081ce3a3": 1, "2adb53ed55": 95, "2e56d84406": 1, "2e90cb285b": 1, "2ef23e0c6e": 2, "3077acede0": 1, "30dbda4d3b": 8, "31a15e5535": 22, "3208bba267": 1, "32262a76ec": 1, "342fb98cc3": 1, "36f12daee3": 1, "37346649ae": 1, "37e27ab591": 1, "38640e965a": 1, "3afe22a59f": 1, "3c20a1704e": 163, "3f03ccfdcb": 1, "40240d0cad": 1, "4231832a19": 1, "437954929d": 1, "44ae78f496": 1, "44e1eb0590": 1, "44e98c5473": 1, "45299db29b": 1, "456c01ba28": 1, "458e71cf68": 1, "45c57fec3f": 1, "46811864e6": 2, "46848aadd7": 2, "46a9f4f78a": 10, "4739d1095b": 38, "47b62409f4": 7, "490d61d4f3": 23, "4c4ff19fcb": 1, "4cb0039df5": 1, "4f44a75f5c": 1, "5022a942f7": 4, "52a9459fad": 1, "5304ae5a01": 1, "53f14ed5e6": 1, "543aba5444": 13, "55373e05dd": 1, "5655576e5d": 1, "569407d70c": 1, "570708220b": 1, "577371a021": 1, "57af4ed9e7": 1, "59fa20a661": 6, "5c5f55e05f": 1, "5cef296b16": 1, "5d7d8f6e60": 1, "5fb4ce1fd8": 9, "613f917e09": 1, "62385c6454": 1, "62f2feeffe": 1, "634698db5f": 50, "65b26cef4b": 1, "6611dabf43": 1, "66804c88b2": 1, "672b030cb8": 1, "672cc10ebf": 1, "68637a74c1": 1, "6b827bb11a": 1, "6b9fa8792c": 1, "6c2c9d21e0": 1, "711a513348": 1, "726d0bb7db": 1, "74d564d598": 1, "753632e764": 1, "7560ad62d9": 1, "7673773534": 1, "776279f472": 27, "7a276057b7": 1, "865d2a56c7": 1, "87d2c58083": 1, "88a9b22855": 1, "88c62f57ee": 1, "8c9d367161": 1, "9377b10969": 1, "9481ad2b62": 31, "96f6f8a817": 1, "9752ec0c24": 1, "98410eb3ac": 1, "9901d58cfd": 1, "999231cc44": 1, "9b25bd247c": 3, "9bdd0a407b": 1, "9cb5b7529c": 1, "9e00206a78": 12, "9e8f5a22a9": 1, "9ea3131c8c": 1, "a19347679e": 8, "a25b59ecd4": 37, "a5d7e2137d": 1, "a5dc419634": 80, "a7af0dc818": 1, "a9e17a2895": 1, "ac9817065c": 2, "aeff00000c": 1, "af842f2461": 1, "af9c60acf2": 1, "b2158243b9": 3, "b28f9305a0": 1, "b3ccff5913": 1, "b45651c753": 1, "b482c8b1f6": 1, "b512505a7b": 1, "b526ce3549": 1, "b57911b938": 1, "b651308b9e": 1, "b7a2ec3b21": 1, "bb838cf16b": 1, "bd1c492bc9": 1, "bd46d95dd9": 1, "be30e13a01": 1, "c09a7d5c2f": 1, "c1ebb47369": 3, "c3a01a6fea": 1, "c3fd188e53": 1, "c7648caf3e": 1, "c7e909dabe": 1, "c860b28e53": 1, "cbcc1bfe23": 1, "cc49d60eb1": 1, "cca9994587": 1, "cdfca24c46": 1, "d0285d51c9": 10, "d1a30e48d5": 1, "d22b0fe87e": 16, "d24b793042": 4, "d405a03580": 1, "d4f1ee2221": 22, "d665c665ea": 3, "d75d11147d": 1, "d77a2ac53a": 1, "d8153921c0": 1, "d8ed4f6a67": 1, "dab54ba446": 1, "db04baf2e6": 1, "db7ae2c1bc": 2, "e177745c60": 3, "e4c15fb981": 19, "e53cef93ab": 1, "e610b7a847": 1, "e6ea188612": 1, "e708c2e1be": 12, "e83c9b4116": 1, "e9bdfc8f1d": 1, "ec99421459": 1, "ee3ae56e98": 1, "f106f0cc08": 1, "f1f2f2619d": 15, "f2f43016f3": 1, "f42a6ef5df": 1, "f520e3b188": 1, "f67a461853": 1, "fae8b22f83": 1, "fc0356807b": 1, "fc0f493f15": 1, "fc70010f4f": 1, "fd4c271910": 1, "fe40e2c3a1": 1}, "2026-09-10": {"0014f58b16": 1, "009b3bca0d": 2, "05816a5153": 1, "05b935e1cc": 1, "06d24527c6": 8, "08248e0037": 1, "0839753672": 5, "0aab549512": 1, "0b49498ce0": 1, "0b63555121": 1, "0b74c85e52": 1, "0cc9495557": 43, "0ea46b9ce2": 1, "0ecb0de9d5": 1, "0ed1a8485b": 1, "0ee407b42a": 76, "116315ebab": 2, "122c90f37c": 1, "1292bb7521": 1, "13546d3ede": 1, "13750291df": 1, "155b091e6c": 1, "157ae34804": 1, "159f869033": 1, "16642f5ec3": 1, "168b36a0a5": 116, "168dc03b6a": 1, "17bc423f36": 1, "19233d61d7": 1, "194579f13b": 1, "1af0f9ab59": 1, "1b7ce69686": 4, "1c933f2239": 1, "1ccf51b6ac": 1, "1d5d7dc5ee": 7, "1ff355c371": 1, "206e0ca52b": 27, "2186e7ab7f": 13, "220f69598f": 1, "2236784afc": 1, "2284f71ec4": 1, "22e6a5200f": 1, "2592797064": 1, "26bcf7ed64": 1, "279b060660": 1, "281e031004": 6, "28c9ccaae4": 3, "29cafb7be2": 1, "2a740471e0": 1, "2a806eb543": 1, "2adb53ed55": 2, "2bfcc59cf5": 1, "2cc0bba4f2": 1, "2f258cbd5f": 1, "301de65ba9": 1, "305cb2a5fb": 1, "30e2e49b5a": 1, "319546e697": 1, "31a15e5535": 1, "31c73e37fc": 1, "342fb98cc3": 2, "355db04cf3": 1, "370cbcde4b": 1, "393ab6b7eb": 1, "3ade6acb71": 1, "3b4e2d221f": 1, "3bbf3bbaa1": 1, "3c20a1704e": 157, "3d516dc3b7": 1, "3e593ee012": 1, "402c4b9df4": 7, "40a2354896": 1, "41a151941a": 1, "45619542ea": 1, "45be607935": 1, "4739d1095b": 6, "47b62409f4": 6, "47fc6399f6": 1, "4bc23f1247": 1, "4ca71813ca": 1, "4cd9dfae0f": 1, "4f1d308998": 1, "503fe454e9": 1, "50e3e2ecc5": 1, "5122b8b19d": 2, "543aba5444": 2, "56d01571c4": 2, "57af4ed9e7": 6, "58a01181b5": 1, "5926bede1a": 1, "5934fc9b2e": 1, "5b292b2539": 3, "5cf3b01819": 1, "5fb4ce1fd8": 25, "606f13cc8c": 1, "6118b11322": 1, "612079b2ab": 1, "61edb1e613": 1, "61f56f0390": 1, "63089b6d30": 1, "634698db5f": 23, "6366e6dc04": 1, "6539a8f173": 5, "66c68c32ed": 1, "677f9a0ea6": 1, "68f8ec309a": 5, "6a4f6a8301": 1, "6c21a27d6d": 1, "6d29cc0e89": 1, "6e1dd758f5": 1, "6f01b48651": 1, "6fc7f2dd21": 1, "72b11a6137": 1, "73b43a2d2b": 1, "73dc7fd176": 1, "741b4a3cc9": 1, "746448fb96": 5, "7525d61920": 5, "774c25f21c": 1, "776279f472": 70, "77c7c0148b": 1, "780c661b59": 1, "7899fd1778": 1, "7bbed15e64": 2, "7e9fb7c0c4": 1, "7f30de01a9": 1, "80096f8eca": 1, "80bdea477b": 1, "82a38104a2": 1, "82da5e37c3": 107, "84454f6d21": 1, "84f1d0db6d": 1, "855ed24a65": 1, "85884dd1b3": 1, "85a4fc80e8": 1, "86676e215b": 1, "889997d0e3": 1, "8bb08f2431": 7, "8c5c975111": 1, "8e0d099a8d": 1, "8f3b6fb65a": 1, "9203129256": 1, "9246c8132a": 1, "9458ea4d2f": 1, "9481ad2b62": 21, "94d53a397f": 35, "95e0754879": 1, "982f2ba316": 1, "99754adbc0": 11, "9a1cd96e09": 1, "9aa034b8c1": 1, "9b16723b7a": 1, "a1112e32cc": 1, "a25b59ecd4": 32, "a3a4e72b9f": 1, "a3dafb074a": 1, "a40dea0728": 1, "ab07df57da": 1, "ac05527b47": 1, "ac206e2370": 1, "ac9817065c": 5, "afb3ff5aa7": 1, "b090399e65": 1, "b14f478576": 2, "b2ea694cce": 1, "b3ae0cb7d8": 1, "b409129949": 1, "b856f59cef": 1, "b94188df6a": 1, "ba0d2d1e74": 1, "bb838cf16b": 1, "bba1d8711a": 1, "bd169e0b50": 1, "bd913b5623": 1, "bf48685abd": 1, "bfbfa1a355": 1, "c001009db8": 2, "c03c3866e9": 1, "c1ebb47369": 30, "c35f03c66e": 6, "c3b309e56c": 7, "c4a92825ee": 1, "c5ec5f4349": 1, "c860b28e53": 4, "c8f445694f": 1, "c917c5a2b9": 1, "c964a120b9": 1, "cca9994587": 1, "cccc86e535": 1, "ccf05a642c": 1, "cf66563e29": 6, "cf7ca9f502": 2, "d0285d51c9": 90, "d1fd85ea91": 1, "d2cc151601": 1, "d36feee0a6": 1, "d3e43cbd76": 1, "d4a8d119a7": 35, "d4f1ee2221": 2, "d50b3e1e1e": 4, "d5d8ba9cb4": 1, "d763a10279": 6, "d7f31ba64c": 1, "d8e17decb1": 1, "da5132c4ac": 1, "dd146a8e74": 1, "dd7daafe34": 1, "dd8372c431": 1, "de83a25e97": 1, "de9f114106": 1, "deda21c929": 1, "deee5a6bbc": 2, "df31d4269b": 1, "dfd316d36a": 1, "e0176e8eeb": 1, "e3d458f3f3": 1, "e4c15fb981": 1, "e4c9662492": 7, "e4d777198b": 1, "e4f2cf5b46": 1, "e663ff9938": 1, "e95c1fa134": 1, "ed76f0e7b7": 32, "ed98b5990a": 1, "ee53179ac0": 1, "ee9c538bbd": 1, "eefdfc7aaa": 1, "ef4ac2a6f5": 1, "f07408c9a0": 1, "f1db8bff81": 1, "f1f2f2619d": 16, "f370f886cf": 1, "f3b59ca8bb": 1, "f48ccac195": 1, "f50514a07f": 1, "f75fa34f5b": 1, "f894f92530": 2, "f9a51930de": 1, "f9d3510db1": 1, "fa5de4f946": 1, "fa662e0112": 1, "fa8826b020": 1, "fbb2bb2a8e": 1, "fbf47e188f": 1, "fc1790935e": 1, "fc57e0371b": 1, "fc6b00f434": 1, "fd5cb631b6": 1, "fdbefff32f": 1, "fea224248c": 1, "ffb0922f1c": 1}, "2026-09-11": {"035545110f": 1, "0a93fb0181": 1, "0b0c99dfb8": 1, "0b1df402e1": 1, "0c4d42c339": 2, "0cc9495557": 43, "0d80900f2e": 1, "0ee407b42a": 77, "0fb646a1d0": 1, "115cf0c59b": 1, "116315ebab": 3, "11ab284107": 1, "129ba29f98": 1, "15f6752e6a": 8, "168b36a0a5": 113, "1768e9fde4": 2, "17b9a80ca3": 1, "1b7ffb5ce5": 1, "1bd47ef9e0": 8, "1d5d7dc5ee": 2, "1dda7d5794": 1, "20d01aac28": 1, "21f9db69e0": 1, "2351058034": 1, "248661ff41": 1, "24901f76a7": 1, "24eba8de59": 1, "256aa5a636": 14, "2588571d24": 3, "27256165e4": 1, "2874742ebe": 1, "29cafb7be2": 2, "2a3a798910": 35, "2a740471e0": 1, "2d4b0d4024": 1, "2dc82c373c": 2, "2e4f927c2b": 1, "30caf54e3b": 1, "3142484c0c": 1, "3145528853": 1, "31f6312c2d": 1, "336b84c507": 1, "33782bc1c2": 1, "33d7a41a82": 1, "34351aeaf6": 1, "356856215b": 1, "35b7a6cd4a": 1, "3bf84dd793": 1, "3c20a1704e": 48, "3d06a2aed3": 1, "402403a6c3": 1, "402c4b9df4": 2, "4157b8ea7a": 1, "432c5226b5": 1, "45807a3648": 1, "45be607935": 10, "47a62ab2ae": 1, "47b62409f4": 1, "4853c19a6d": 3, "48f124c605": 1, "499e83bda9": 1, "4a25c58be9": 1, "4b77a978be": 1, "4dd5d5456b": 1, "4ef8154d6f": 1, "504d28dfb1": 1, "5122b8b19d": 1, "53af4615e4": 12, "54bc608b95": 1, "550a231972": 1, "56d01571c4": 2, "5813d41fc1": 1, "591ec51dee": 1, "595ec7c670": 10, "5a36e864a4": 1, "5adca184f4": 1, "5b292b2539": 18, "5b596fcfa4": 1, "5ba5457af7": 1, "5c127a9242": 1, "5e86b056ac": 1, "5e8f0efe36": 18, "5ef528b074": 1, "60fe2344df": 1, "64cead09e5": 1, "66021d8505": 13, "66799c3aca": 1, "6782e2d263": 1, "67d54b3b0e": 5, "68e76c7f7d": 1, "68f8ec309a": 26, "6a3a65f5f4": 1, "6a5595710c": 1, "6c7eeac29c": 1, "6e11b01ff5": 1, "6e5b392c05": 1, "6fd4e24587": 1, "71352cc946": 14, "71b266a911": 1, "721dbd64c6": 1, "729ed77b38": 1, "746448fb96": 1, "75c75025c0": 14, "7622b1627e": 1, "76a53ff709": 1, "776279f472": 21, "77c7c0148b": 3, "79f9685e74": 1, "7a1e953ad1": 1, "7a4ae87af1": 1, "7abfe9440c": 1, "7bc1ae5961": 1, "7bcb4e8654": 1, "7c3065fe21": 1, "7c63ad4dfc": 1, "7c804fe28b": 1, "7c9c8f3824": 1, "7de4c6922f": 1, "7de80bb1a2": 1, "7e99f9589a": 1, "80792cfc4d": 8, "8125df6b44": 1, "826baf585f": 1, "8327255491": 1, "83491bd157": 1, "868f1f2b73": 1, "87ba59bdcd": 1, "8a438f1a37": 1, "8a6fb90873": 2, "8bb08f2431": 33, "8c19e1c9f1": 1, "8d403e651e": 1, "8db0b2bb62": 1, "8e1ad20c32": 1, "8eae812773": 2, "8eea7ee858": 1, "8f0f4ffc50": 1, "8f3b6fb65a": 5, "8f640e4966": 1, "9071999c34": 1, "910c26b887": 1, "9344957026": 21, "94d53a397f": 1, "94e52fd28a": 1, "957dc31b18": 1, "9601ea0a75": 1, "99754adbc0": 34, "9bae7e3a59": 3, "9f2ac547ef": 1, "a04d8240eb": 1, "a05150bed0": 1, "a086848e4e": 1, "a0da214cb3": 1, "a25b59ecd4": 1, "a6b8543443": 1, "a963946b30": 1, "a9c72d714c": 1, "ac3ec56807": 1, "ad23768f56": 1, "ad5f6b3833": 1, "ad99a14c2a": 31, "aee381861e": 3, "af7d176715": 1, "afb3ff5aa7": 5, "b21127d7c5": 1, "b2479f0b0d": 1, "b2b9e17786": 1, "b2fccfa867": 1, "b3220370ee": 1, "b39df202dc": 1, "b3bb62e28e": 4, "b4f637a27a": 1, "b6b62a9c18": 4, "b9a72f8cb8": 1, "b9ef0a25ad": 4, "ba59bc6905": 1, "bb109a05e4": 1, "bb7ed90ed7": 1, "bc4e332223": 1, "bcbcca1922": 15, "bdcefd300f": 1, "be12d198a1": 1, "be80799865": 1, "bedc236f54": 14, "bede43373c": 1, "beebf381b5": 1, "bef3a880b3": 4, "bf0bf18734": 1, "c001009db8": 7, "c1ebb47369": 30, "c2a5c0b3a4": 37, "c31c9228a4": 7, "c35f03c66e": 6, "c6a217bb8c": 1, "c78dc97385": 1, "c7daf2dd8e": 1, "c918361f9c": 1, "cb63dc56f2": 1, "cc07868524": 1, "cddc643ac1": 1, "ceaef92eb5": 1, "cf66563e29": 3, "cf7ca9f502": 3, "cfdd1d3393": 1, "d0229b64bc": 1, "d033ec2ac4": 1, "d04d1be305": 1, "d178dca2eb": 1, "d1b865b8fc": 1, "d24b793042": 12, "d3498ac87b": 1, "d474893b1c": 14, "d4a8d119a7": 63, "d4cb88ffff": 1, "d50b3e1e1e": 27, "d53b461882": 58, "d5db62ce26": 2, "d763a10279": 3, "d777f2351a": 1, "d7cdb9e5fc": 1, "d7f57abd28": 1, "d9af37ea86": 1, "da09663174": 1, "dba5379883": 1, "dc2dbf03cf": 1, "dc623b4f3a": 15, "dd02a129f4": 1, "dd5e45de47": 1, "dd7daafe34": 1, "dd8372c431": 1, "de5892efba": 1, "de9f114106": 1, "e4c9662492": 28, "e514526bf3": 1, "e52ce41356": 1, "e5fc93398c": 1, "e8b1d1f3b3": 1, "eacdff4941": 1, "eafa535033": 1, "ed76f0e7b7": 70, "ef80ef9367": 3, "f046ca4c3d": 1, "f0e41c71bf": 1, "f131b77f48": 1, "f1e8f23f2d": 3, "f1f2f2619d": 1, "f2edbca7f1": 1, "f3670ef6f8": 1, "f3ad921f82": 1, "f3e8dece8f": 1, "f5890330cd": 1, "f67dff794a": 1, "f7cae527cf": 1, "f842200f83": 5, "f94f9f7d8f": 1, "f9a51930de": 1, "fa01156b51": 1, "fa4b40b807": 1, "fa6e5b4129": 1, "fc885aeb24": 1}}
# --- nbmine:history:end ---

OUT = "/kaggle/working"


def lineage_sig(steps, seat):
    """Opening signature: hash of the seat's first 24 steps of actions."""
    stream = []
    for t in range(24):
        entry = steps[t + 1][seat] if len(steps) > t + 1 else None
        stream.append((entry or {}).get("action") or {})
    return hashlib.sha256(json.dumps(stream, sort_keys=True).encode()).hexdigest()[:10]


def mine_date(date, files):
    census = []
    for k, path in enumerate(sorted(files)):
        eid = int(os.path.basename(path).split(".")[0])
        try:
            rep = json.load(open(path))
        except Exception as e:
            print(f"[{k}] {eid} parse fail {e}", flush=True)
            continue
        steps = rep.get("steps") or []
        if len(steps) < 700:
            continue
        shops = steps[-1][0]["observation"]["town"]["unlocked_shops"]
        census.append({
            "episode": eid, "date": date,
            "town": shops[0] if shops else None,
            "rewards": rep.get("rewards") or [None, None],
            "teams": (rep.get("info") or {}).get("TeamNames"),
            "lineages": [lineage_sig(steps, 0), lineage_sig(steps, 1)],
        })
        if k % 100 == 0:
            print(f"[{k}/{len(files)}] ep{eid}", flush=True)
    with open(f"{OUT}/census-{date}.json", "w") as fh:
        json.dump(census, fh)
    print(f"{date}: {len(census)} episodes", flush=True)
    return census


def render_trend(history):
    """Stacked-area occupancy chart, rendered inline + saved as HTML."""
    import plotly.graph_objects as go
    import plotly.io as pio

    pio.renderers.default = "notebook"  # embed plotly.js in the cell output

    dates = sorted(history)
    if not dates:
        return
    # Pick the majors: overall presence, weighted toward the most recent day.
    total = {}
    for d in dates:
        n = sum(history[d].values()) or 1
        for lin, k in history[d].items():
            total[lin] = total.get(lin, 0) + k / n
    recent = history[dates[-1]]
    rn = sum(recent.values()) or 1
    score = {lin: total[lin] + 3 * recent.get(lin, 0) / rn for lin in total}
    majors = sorted(score, key=score.get, reverse=True)[:9]

    fig = go.Figure()
    ns = {d: sum(history[d].values()) or 1 for d in dates}
    acc = {d: 0.0 for d in dates}
    for m in majors:
        ys = [history[d].get(m, 0) / ns[d] * 100 for d in dates]
        for d, y in zip(dates, ys):
            acc[d] += y
        fig.add_trace(go.Scatter(
            x=dates, y=ys, name=NOTEBOOK_LABELS.get(m, m), stackgroup="one",
            mode="lines", hovertemplate="%{y:.1f}%"))
    fig.add_trace(go.Scatter(
        x=dates, y=[max(0.0, 100.0 - acc[d]) for d in dates], name="others",
        stackgroup="one", mode="lines", hovertemplate="%{y:.1f}%"))
    fig.update_layout(
        title="Kaggriculture top-band lineage occupancy (seats/day)<br>"
              "<sup>seats sampled: "
              + ", ".join(f"{d[5:]}:{ns[d]}" for d in dates) + "</sup>",
        yaxis=dict(range=[0, 100], title="share of seats", ticksuffix="%"),
        hovermode="x unified", height=640,
        # 凡例はプロット領域の外(下)。yanchor=top + 十分な下余白で重なり回避
        legend=dict(orientation="h", yanchor="top", y=-0.10, x=0,
                    font=dict(size=11)),
        margin=dict(t=80, b=30))
    fig.write_html(f"{OUT}/census_trend.html", include_plotlyjs="cdn")
    print(f"wrote {OUT}/census_trend.html", flush=True)
    fig.show()


def main():
    ds = []
    for pat in ("/kaggle/input/kaggriculture-episodes-*",
                "/kaggle/input/*/kaggriculture-episodes-*",
                "/kaggle/input/*/*/kaggriculture-episodes-*"):
        ds += [d for d in glob.glob(pat) if os.path.isdir(d)]
    history = {d: dict(c) for d, c in HISTORY.items()}
    for d in sorted(set(ds)):
        m = re.search(r"episodes-(\d{4}-\d{2}-\d{2})$", d)
        if not m:
            continue
        files = glob.glob(f"{d}/*.json") or glob.glob(f"{d}/*/*.json")
        print(f"{d}: {len(files)} files", flush=True)
        census = mine_date(m.group(1), files)
        counts = {}
        for row in census:
            for lin in row["lineages"]:
                counts[lin] = counts.get(lin, 0) + 1
        history[m.group(1)] = counts
    render_trend(history)


main()
